In [1]:
#!/usr/bin/env python3
from __future__ import annotations

import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import optuna
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

DATASET = "Tobi-Bueck/customer-support-tickets"

def read_idx(path: Path) -> np.ndarray:
    return np.loadtxt(path, dtype=np.int64)

def md5(s: str) -> str:
    return hashlib.md5(s.encode("utf-8", errors="ignore")).hexdigest()

def split_df(df: pd.DataFrame, repo_root: Path) -> dict[str, pd.DataFrame]:
    train_idx = read_idx(repo_root / "data" / "train_idx.txt")
    val_idx = read_idx(repo_root / "data" / "val_idx.txt")
    test_idx = read_idx(repo_root / "data" / "test_idx.txt")
    return {
        "train": df.iloc[train_idx].copy(),
        "val": df.iloc[val_idx].copy(),
        "test": df.iloc[test_idx].copy(),
    }

def run_optuna_baseline(splits: dict[str, pd.DataFrame]):
    def make_text(d: pd.DataFrame) -> pd.Series:
        return (d["subject"].fillna("") + "\n\n" + d["body"].fillna("")).astype(str)

    # Готовим данные
    X_train_text = make_text(splits["train"])
    X_val_text = make_text(splits["val"])
    X_test_text = make_text(splits["test"])

    # Фиксируем векторизатор (можно тоже сунуть в Optuna, но это замедлит процесс)
    vec = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), min_df=5)
    Xtr = vec.fit_transform(X_train_text)
    Xval = vec.transform(X_val_text)
    Xte = vec.transform(X_test_text)

    targets = ["queue", "priority", "type"]
    y_train = {t: splits["train"][t].astype(str).values for t in targets}
    y_val = {t: splits["val"][t].astype(str).values for t in targets}
    y_test = {t: splits["test"][t].astype(str).values for t in targets}

    def objective(trial):
        param_c = trial.suggest_float("C", 0.01, 10.0, log=True)
        
        # Чтобы избежать ошибки, используем только squared_hinge при dual=False
        # Либо, если очень хочется hinge, придется ставить dual=True (но это медленно)
        param_loss = "squared_hinge" 
        is_dual = False 
        
        results = {}
        for t in targets:
            clf = LinearSVC(
                C=param_c, 
                loss=param_loss, 
                dual=is_dual, 
                class_weight='balanced', 
                max_iter=1000,
                random_state=42
            )
            clf.fit(Xtr, y_train[t])
            pred = clf.predict(Xval)
            
            if t == "queue":
                results[t] = f1_score(y_val[t], pred, average="macro")
            else:
                results[t] = accuracy_score(y_val[t], pred)

        return 0.70 * results["queue"] + 0.15 * results["priority"] + 0.15 * results["type"]
        
    print("\n=== ЗАПУСК OPTUNA ДЛЯ LINEARSVC (30 МИНУТ) ===")
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, timeout=60*30, n_jobs=1) # n_jobs=1 для стабильности LinearSVC

    print("\n=== ЛУЧШИЕ ПАРАМЕТРЫ ===")
    print(study.best_params)
    
    # Финальный прогон на тесте
    print("\n=== ТЕСТИРОВАНИЕ ЛУЧШЕЙ МОДЕЛИ ===")
    best_params = study.best_params
    final_results = {}
    
    for t in targets:
        clf = LinearSVC(
            **best_params, 
            dual=False, 
            class_weight='balanced', 
            max_iter=2000, 
            random_state=42
        )
        clf.fit(Xtr, y_train[t])
        pred = clf.predict(Xte)
        
        acc = accuracy_score(y_test[t], pred)
        if t == "queue":
            mf1 = f1_score(y_test[t], pred, average="macro")
            final_results["q_f1"] = mf1
            print(f"{t}: acc={acc:.4f} macro_f1={mf1:.4f}")
        else:
            final_results[f"{t}_acc"] = acc
            print(f"{t}: acc={acc:.4f}")

    final_score = 0.70 * final_results["q_f1"] + 0.15 * final_results["priority_acc"] + 0.15 * final_results["type_acc"]
    print(f"\nFINAL TEST SCORE: {final_score:.4f}")

def main():
    repo_root = Path(".").resolve()
    if not (repo_root / "data" / "train_idx.txt").exists():
        raise SystemExit("Запусти из корня репо, где есть папка data/ с *_idx.txt")

    print("Loading dataset:", DATASET)
    ds = load_dataset(DATASET)["train"]
    df = ds.to_pandas()

    splits = split_df(df, repo_root)

    # На всякий случай чистим пропуски в таргетах
    for k in splits:
        for t in ["queue", "priority", "type"]:
            splits[k][t] = splits[k][t].fillna("Unknown")
    
    # Запуск тюнинга
    run_optuna_baseline(splits)

if __name__ == "__main__":
    main()

Loading dataset: Tobi-Bueck/customer-support-tickets


[I 2026-03-14 19:25:18,619] A new study created in memory with name: no-name-c85bbc43-e813-4cc3-b671-e195c22d0932



=== ЗАПУСК OPTUNA ДЛЯ LINEARSVC (30 МИНУТ) ===


[I 2026-03-14 19:26:14,266] Trial 0 finished with value: 0.8426416164641534 and parameters: {'C': 2.9385877961538793}. Best is trial 0 with value: 0.8426416164641534.
[I 2026-03-14 19:26:46,682] Trial 1 finished with value: 0.7090131366119504 and parameters: {'C': 0.052244663797880814}. Best is trial 0 with value: 0.8426416164641534.
[I 2026-03-14 19:27:39,814] Trial 2 finished with value: 0.8424984982432109 and parameters: {'C': 2.4600438478381297}. Best is trial 0 with value: 0.8426416164641534.
[I 2026-03-14 19:28:12,431] Trial 3 finished with value: 0.7105183011158046 and parameters: {'C': 0.05386484902181327}. Best is trial 0 with value: 0.8426416164641534.
[I 2026-03-14 19:28:44,204] Trial 4 finished with value: 0.6393600662944219 and parameters: {'C': 0.01864267952584104}. Best is trial 0 with value: 0.8426416164641534.
[I 2026-03-14 19:29:16,242] Trial 5 finished with value: 0.6847387056241854 and parameters: {'C': 0.03823618546669057}. Best is trial 0 with value: 0.84264161646

KeyboardInterrupt: 